class VolatilityStrategy_1(bt.Strategy):
    params = (
        ('vol_period', 20),  # Period for volatility calculation
        ('sma_period', 20),  # Simple Moving Average period
        ('risk_pct', 0.02),  # Risk per trade (2% of capital)
        ('stop_loss_atr', 1.5),  # Stop loss at 1.5x ATR
        ('take_profit_atr', 2.5)  # Take profit at 2.5x ATR
    )

    def __init__(self):
        # Indicators
        self.volatility = bt.indicators.StandardDeviation(self.data.close, period=self.params.vol_period)
        self.sma = bt.indicators.SimpleMovingAverage(self.data.close, period=self.params.sma_period)
        self.atr = bt.indicators.ATR(self.data, period=14)  # ATR for volatility-based stop loss

    def next(self):
        position_size = self.broker.get_cash() * self.params.risk_pct / self.atr[0]  # Risk-based position sizing

        # Buy Condition: Volatility increasing & Price above SMA
        if not self.position and self.volatility[0] > self.volatility[-1] and self.data.close[0] > self.sma[0]:
            self.buy(size=position_size)
            self.stop_price = self.data.close[0] - (self.params.stop_loss_atr * self.atr[0])
            self.take_profit_price = self.data.close[0] + (self.params.take_profit_atr * self.atr[0])

        # Sell Condition: Volatility decreasing & Price below SMA
        elif not self.position and self.volatility[0] < self.volatility[-1] and self.data.close[0] < self.sma[0]:
            self.sell(size=position_size)
            self.stop_price = self.data.close[0] + (self.params.stop_loss_atr * self.atr[0])
            self.take_profit_price = self.data.close[0] - (self.params.take_profit_atr * self.atr[0])

        # Check Stop-Loss and Take-Profit
        if self.position:
            if (self.position.size > 0 and self.data.close[0] <= self.stop_price) or \
               (self.position.size < 0 and self.data.close[0] >= self.stop_price):
                self.close()  # Stop-loss hit

            if (self.position.size > 0 and self.data.close[0] >= self.take_profit_price) or \
               (self.position.size < 0 and self.data.close[0] <= self.take_profit_price):
                self.close()  # Take-profit hit

# Backtest Setup
cerebro = bt.Cerebro()
data_feed = bt.feeds.PandasData(dataname=apple_stock_data)
cerebro.adddata(data_feed)
cerebro.addstrategy(VolatilityStrategy_1)
cerebro.run()
cerebro.plot(volume=False, iplot=False)



In [ ]:
class VolatilityStrategy(bt.Strategy):
    params = (('vol_period', 20),)

    def __init__(self):
        self.volatility = bt.indicators.StandardDeviation(self.data.close, period=self.params.vol_period)
        self.sma = bt.indicators.SimpleMovingAverage(self.data.close, period=20)

    def next(self):
        if self.volatility[0] > self.volatility[-1] and self.data.close[0] > self.sma[0]:
            self.buy()
        elif self.volatility[0] < self.volatility[-1] and self.data.close[0] < self.sma[0]:
            self.sell()

# Setup backtest
cerebro = bt.Cerebro()
data_feed = bt.feeds.PandasData(dataname=apple_stock_data)
cerebro.adddata(data_feed)
cerebro.addstrategy(VolatilityStrategy)
cerebro.run()
cerebro.plot(style='candle', volume=False)


import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Function to calculate ATR
def calculate_atr(data, window=14):
    high_low = data['High'] - data['Low']
    high_close = abs(data['High'] - data['Close'].shift())
    low_close = abs(data['Low'] - data['Close'].shift())

    # True Range (TR)
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

    # Average True Range (ATR)
    atr = tr.rolling(window=window).mean()
    return atr

# Fetch stock data
ticker = "AAPL"  # You can change to any stock symbol
data = yf.download(ticker, start="2022-01-01", end="2023-12-31")

# Calculate ATR
data['ATR'] = calculate_atr(data)

# Plot ATR and closing price
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot stock price
ax1.set_xlabel("Date")
ax1.set_ylabel("Stock Price", color="blue")
ax1.plot(data['Close'], label="Closing Price", color="blue", alpha=0.6)
ax1.tick_params(axis='y', labelcolor="blue")

# Plot ATR
ax2 = ax1.twinx()
ax2.set_ylabel("ATR", color="red")
ax2.plot(data['ATR'], label="ATR (14-day)", color="red", linestyle="dashed", alpha=0.75)
ax2.tick_params(axis='y', labelcolor="red")

# Add legend and title
fig.suptitle(f"{ticker} Stock Price and ATR Indicator", fontsize=14)
fig.tight_layout()
plt.show()
